# Audit de la méthode CoteScope robuste
Archives cotes-value présélectionnées, 5–7 octobre 2026. Replay chronologique ; aucun champ de résultat n’entre dans les décisions. Les trois jours nominaux contiennent une seule journée avec candidats contrôlés admissibles. Aucun avantage de rendement démontré.

In [ ]:
from pathlib import Path
import gzip, json, collections, hashlib, os, subprocess
repo = Path(os.environ.get("COTESCOPE_REPO", "/workspace/Cotescope"))
data = Path(os.environ.get("COTESCOPE_AUDIT_DATA", "/tmp/cotescope-method-data"))
revision = "7fed7e82988c55e3082226626818081508067ddc"
assert data.exists(), "Extraire les archives de cette révision selon docs/METHOD.md"


In [ ]:
profile = []
for file in sorted(data.glob("*.jsonl.gz")):
    rows = [json.loads(line) for line in gzip.decompress(file.read_bytes()).splitlines()]
    keys = [json.dumps([r.get(k) for k in ["detecte", "bookmaker", "match_id", "marche", "periode", "ligne", "issue", "joueur", "reference"]]) for r in rows]
    profile.append({"day": file.name[:10], "rows": len(rows), "duplicate_detection_reference_keys": len(keys)-len(set(keys)), "missing_reference_time": sum(not r.get("lu_reference") for r in rows), "sources": dict(collections.Counter(r["reference"] for r in rows)), "controls": dict(collections.Counter((r.get("controle") or {}).get("statut", "absent") for r in rows)), "sha256": hashlib.sha256(file.read_bytes()).hexdigest()})
print(json.dumps(profile, indent=2, ensure_ascii=False))


In [ ]:
output = data / "reproduced-comparison.json"
subprocess.run(["node", "scripts/compare-methods.cjs", "--data-dir", str(data), "--out", str(output), "--source-revision", revision], cwd=repo, check=True)
report = json.loads(output.read_text())
published = json.loads((repo / "src/data/method-comparison.json").read_text())
for field in ["quality", "sources", "settlementsSha256", "baseline", "cotescope", "perDay", "rejected"]:
    assert report[field] == published[field], field
print("Résultats reproduits ; lire les limites avant interprétation.")


## Risques à conserver dans la conclusion
- Présélection en amont : les prix non publiés manquent (sévérité élevée).
- 40 887 références brutes avec marge sur 44 722 lignes, exclues de la comparaison.
- Répétitions de snapshots et simulations : grain de décision = candidat, première prise admissible ; une prise par match pour CoteScope.
- 12 résultats manquants dans la référence, cohorte CoteScope d’une seule prise réglée : comparaison de ROI non concluante.
- Dernière journée partielle ; aucune validation prospective indépendante.
- Temps de lecture publiés, pas preuve de la dernière mise à jour du prix chez l’opérateur.
- Politique fixe et heuristique ; sa marge de stress n’est pas une borne statistique.
